In [1]:
import brightway2 as bw
from brightway2 import *
import time
import numpy as np
import openpyxl
import os
import pandas as pd
import glob

In [2]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [3]:
bw.projects.set_current('Prospective LCA v2') # set current project

In [ ]:
databaseNames = bw.databases
myDatabaseNames = []
for databaseName in databaseNames:
    if 'Abhi' in databaseName:
        myDatabaseNames.append(databaseName)

In [ ]:
myDatabaseNames

In [ ]:
methodsDict = {'climate change':                ('IPCC 2013', 'climate change', 'GWP 100a, incl. H and bio CO2'),
            'acidification':                    ('EF v3.0 no LT', 'acidification no LT', 'accumulated exceedance (ae) no LT'),
            'eutrophication freshwater':        ('EF v3.0 no LT', 'eutrophication: freshwater no LT', 'fraction of nutrients reaching freshwater end compartment (P) no LT'),
            'eutrophication marine':            ('EF v3.0 no LT', 'eutrophication: marine no LT', 'fraction of nutrients reaching marine end compartment (N) no LT'),
            'eutrophication terrestrial':       ('EF v3.0 no LT', 'eutrophication: terrestrial no LT', 'accumulated exceedance (AE)  no LT'),
            'photochemical ozone formation':    ('EF v3.0 no LT', 'photochemical ozone formation: human health no LT', 'tropospheric ozone concentration increase no LT'),
            'particulate matter':               ('EF v3.0 no LT', 'particulate matter formation no LT', 'impact on human health no LT'),
            'ozone depletion':                  ('EF v3.0 no LT', 'ozone depletion no LT', 'ozone depletion potential (ODP)  no LT'),
            'ecotoxicity freshwater':           ('EF v3.0 no LT', 'ecotoxicity: freshwater no LT', 'comparative toxic unit for ecosystems (CTUe)  no LT'),
            'human toxicity carcinogenic':      ('EF v3.0 no LT', 'human toxicity: carcinogenic no LT', 'comparative toxic unit for human (CTUh)  no LT'),
            'human toxicity non-carcinogenic':  ('EF v3.0 no LT', 'human toxicity: non-carcinogenic no LT', 'comparative toxic unit for human (CTUh)  no LT'),
            'ionising radiation':               ('EF v3.0 no LT', 'ionising radiation: human health no LT', 'human exposure efficiency relative to u235 no LT'), 
            'resource use fossils':             ('EF v3.0 no LT', 'energy resources: non-renewable no LT', 'abiotic depletion potential (ADP): fossil fuels no LT'),
            'resource use mineral and metals':  ('EF v3.0 no LT', 'material resources: metals/minerals no LT', 'abiotic depletion potential (ADP): elements (ultimate reserves) no LT'),
            'land use':                         ('EF v3.0 no LT', 'land use no LT', 'soil quality index no LT'),
            'water use':                        ('EF v3.0 no LT', 'water use no LT', 'user deprivation potential (deprivation-weighted water consumption) no LT')
            }    

In [ ]:
methodsList = []
hydrogenResultsFileNames = []
carbonDioxideResultsFileNames = []
ammoniaResultsFileNames = []
methanolResultsFileNames = []
ethyleneResultsFileNames = []
for keys, values in methodsDict.items():  
    methodsList.append(values)
    hydrogenResultsFileNames.append('hydrogen ' + keys + ' results.xlsx')
    carbonDioxideResultsFileNames.append('carbon dioxide ' + keys + ' results.xlsx')
    ammoniaResultsFileNames.append('ammonia ' + keys + ' results.xlsx')
    methanolResultsFileNames.append('methanol ' + keys + ' results.xlsx')
    ethyleneResultsFileNames.append('ethylene ' + keys + ' results.xlsx')

allResultsFileNames = hydrogenResultsFileNames + carbonDioxideResultsFileNames + ammoniaResultsFileNames + methanolResultsFileNames + ethyleneResultsFileNames

In [ ]:
def lca_calculations(activities, methodsList):
    results = np.zeros((len(activities), len(methodsList)))
    lca = LCA({activities[0] : 1}, method = methodsList[0]) # LCA object which will do all calculating
    lca.lci() # calculate inventory once to load all database data
    lca.decompose_technosphere() # keep the LU factorized matrices for faster calculations
    lca.lcia() # load method data
    characterizationMatrices = []
    
    for method in methodsList:
        lca.switch_method(method)
        characterizationMatrices.append(lca.characterization_matrix.copy())
        
    for first, activity in enumerate(activities):
        lca.redo_lci({activity : 1})
        
        for second, matrix in enumerate(characterizationMatrices):
            results[first, second] = (matrix * lca.inventory).sum()
            
    return results

In [ ]:
def check_or_create_excel_file(filePath):
    if not os.path.exists(filePath):
        wb = openpyxl.Workbook()
        wb.save(filePath)
        print(f'Excel file created at {filePath}')

In [ ]:
def delete_first_sheet_from_excel_files(filePath):
    wb = openpyxl.load_workbook(filePath)
    if wb.sheetnames[0] == 'Sheet':  # check if the workbook has any sheets
        firstSheet = wb.sheetnames[0]  # get the name of the first sheet
        wb.remove(wb[firstSheet])  # remove the first sheet
        wb.save(filePath)

In [ ]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'hydrogen, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(hydrogenResultsFileNames)):
        hydrogenResultsFilePath = os.path.join('..', 'Results', 'Hydrogen', hydrogenResultsFileNames[i])
        check_or_create_excel_file(hydrogenResultsFilePath)
        with pd.ExcelWriter(hydrogenResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

In [ ]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'carbon dioxide, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(carbonDioxideResultsFileNames)):
        carbonDioxideResultsFilePath = os.path.join('..', 'Results', 'Carbon dioxide', carbonDioxideResultsFileNames[i])
        check_or_create_excel_file(carbonDioxideResultsFilePath)
        with pd.ExcelWriter(carbonDioxideResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

In [ ]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'ammonia, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(ammoniaResultsFileNames)):
        ammoniaResultsFilePath = os.path.join('..', 'Results', 'Ammonia', ammoniaResultsFileNames[i])
        check_or_create_excel_file(ammoniaResultsFilePath)
        with pd.ExcelWriter(ammoniaResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

In [ ]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'methanol, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(methanolResultsFileNames)):
        methanolResultsFilePath = os.path.join('..', 'Results', 'Methanol', methanolResultsFileNames[i])
        check_or_create_excel_file(methanolResultsFilePath)
        with pd.ExcelWriter(methanolResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

In [ ]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'ethylene, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(ethyleneResultsFileNames)):
        ethyleneResultsFilePath = os.path.join('..', 'Results', 'Ethylene', ethyleneResultsFileNames[i])
        check_or_create_excel_file(ethyleneResultsFilePath)
        with pd.ExcelWriter(ethyleneResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

In [ ]:
allResultsFileNames = glob.glob(os.path.join('..', 'Results', '**/*.xls*'), recursive = True)
for filePath in allResultsFileNames:
    delete_first_sheet_from_excel_files(filePath)